In [1]:
!nvidia-smi

Mon Jun  8 08:08:01 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.159.03             Driver Version: 580.159.03     CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  NVIDIA H100 80GB HBM3          On  |   00000000:65:00.0 Off |                  Off |
| N/A   34C    P0             75W /  700W |       0MiB /  81559MiB |      0%      Default |
|                                         |                        |             Disabled |
+-----------------------------------------+-----

In [ ]:
!kill -9 4100      

In [1]:
!pip install -U pip
!pip install transformers==4.46.3 accelerate bitsandbytes peft trl datasets huggingface_hub

In [2]:
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig

# 추천받은 3B 파라미터의 Instruct 모델을 사용합니다.
model_id = "Qwen/Qwen2.5-3B-Instruct"

# H100은 매우 강력하지만, 일반적인 실습 환경(VRAM 절약)을 가정하여 4-bit 양자화 로드를 적용합니다.
bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_use_double_quant=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.bfloat16 # H100은 bfloat16을 완벽히 지원합니다.
)

tokenizer = AutoTokenizer.from_pretrained(model_id)
tokenizer.pad_token = tokenizer.eos_token

model = AutoModelForCausalLM.from_pretrained(
    model_id,
    quantization_config=bnb_config,
    device_map="auto"
)

Loading checkpoint shards:   0%|          | 0/2 [00:00<?, ?it/s]

In [4]:
from datasets import load_dataset

# 범용 비즈니스 및 지시어 데이터셋 (KoAlpaca) 로드
dataset = load_dataset("beomi/KoAlpaca-v1.1a", split="train")

# 모델이 알아들을 수 있도록 Qwen Chat Template 적용 함수
def format_prompt(example):
    messages = [
        {"role": "system", "content": "You are a helpful AI assistant. Respond in Korean."},
        {"role": "user", "content": example["instruction"]}
    ]
    
    # 1. 챗 템플릿 적용
    text = tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
    
    # 2. [방어 코드] 만약 버그로 인해 리스트(숫자)가 반환되었다면 강제로 문자열로 디코딩
    if isinstance(text, list):
        text = tokenizer.decode(text)
        
    # 3. 정답(output)과 종료 토큰 추가
    text += example["output"] + tokenizer.eos_token
    return {"text": text}

# 실습 시간 단축을 위해 2,000건의 데이터만 샘플링하여 훈련 세트로 구성
train_dataset = dataset.map(format_prompt)
print("\n[전처리된 데이터 예시]")
print(train_dataset[0]["text"])


[전처리된 데이터 예시]
<|im_start|>system
You are a helpful AI assistant. Respond in Korean.<|im_end|>
<|im_start|>user
양파는 어떤 식물 부위인가요? 그리고 고구마는 뿌리인가요?<|im_end|>
<|im_start|>assistant
양파는 잎이 아닌 식물의 줄기 부분입니다. 고구마는 식물의 뿌리 부분입니다. 

식물의 부위의 구분에 대해 궁금해하는 분이라면 분명 이 질문에 대한 답을 찾고 있을 것입니다. 양파는 잎이 아닌 줄기 부분입니다. 고구마는 다른 질문과 답변에서 언급된 것과 같이 뿌리 부분입니다. 따라서, 양파는 식물의 줄기 부분이 되고, 고구마는 식물의 뿌리 부분입니다.

 덧붙이는 답변: 고구마 줄기도 볶아먹을 수 있나요? 

고구마 줄기도 식용으로 볶아먹을 수 있습니다. 하지만 줄기 뿐만 아니라, 잎, 씨, 뿌리까지 모든 부위가 식용으로 활용되기도 합니다. 다만, 한국에서는 일반적으로 뿌리 부분인 고구마를 주로 먹습니다.<|im_end|>


In [5]:
from peft import LoraConfig, prepare_model_for_kbit_training
from trl import SFTTrainer, SFTConfig

# k-bit 학습을 위한 준비 및 메모리 누수 방지(필수)
model = prepare_model_for_kbit_training(model)
model.config.use_cache = False  # 학습 중 메모리 터짐 방지

lora_config = LoraConfig(
    r=8,
    lora_alpha=16,
    target_modules=["q_proj", "k_proj", "v_proj", "o_proj"],
    lora_dropout=0.05,
    bias="none",
    task_type="CAUSAL_LM"
)

# 최신 trl 라이브러리에 맞춘 SFTConfig (OOM 방지 설정 포함)
training_args = SFTConfig(
    output_dir="./qwen-koalpaca-lora",
    per_device_train_batch_size=4,   # 4 또는 8 유지
    gradient_accumulation_steps=4,
    gradient_checkpointing=True,     # [핵심] VRAM 폭발 방지
    learning_rate=2e-4,
    logging_steps=10,
    # max_steps=100,                 # 전체 학습을 위해 주석 처리
    num_train_epochs=1,              # 전체 데이터 1에폭 학습
    optim="paged_adamw_8bit",        # 8bit 옵티마이저로 VRAM 절약
    fp16=False,
    bf16=True,                       # H100 전용 bfloat16 가속
    dataset_text_field="text", 
    max_seq_length=1024,             # 문장 최대 길이 1024 고정
    packing=False                    # 텍스트 이어붙이기 방지
)

trainer = SFTTrainer(
    model=model,
    train_dataset=train_dataset,
    args=training_args,
    peft_config=lora_config
)

In [ ]:
trainer.train()

wandb: WARNING The `run_name` is currently set to the same value as `TrainingArguments.output_dir`. If this was not intended, please specify a different run name by setting the `TrainingArguments.run_name` parameter.
wandb: (1) Create a W&B account
wandb: (2) Use an existing W&B account
wandb: (3) Don't visualize my results


wandb: Enter your choice:  3


wandb: You chose "Don't visualize my results"
wandb: Using W&B in offline mode.
wandb: W&B API key is configured. Use `wandb login --relogin` to force relogin


/usr/local/lib/python3.11/dist-packages/torch/utils/checkpoint.py:295: FutureWarning: `torch.cpu.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cpu', args...)` instead.
  with torch.enable_grad(), device_autocast_ctx, torch.cpu.amp.autocast(**ctx.cpu_autocast_kwargs):  # type: ignore[attr-defined]


Step,Training Loss
10,2.108800
20,1.827100
30,1.757500
40,1.740800
50,1.686800
60,1.654100
70,1.680400
80,1.716200
90,1.662200
100,1.659000


In [ ]:
from peft import PeftModel

del model
del trainer
torch.cuda.empty_cache()

# 원본 Base 모델을 양자화 없이 FP16으로 다시 로드
base_model = AutoModelForCausalLM.from_pretrained(
    model_id,
    low_cpu_mem_usage=True,
    return_dict=True,
    torch_dtype=torch.float16,
    device_map="auto"
)

# 방금 학습한 최신 체크포인트의 LoRA 가중치를 불러와 덧씌움
model = PeftModel.from_pretrained(base_model, "./qwen-koalpaca-lora/checkpoint-100")

# 가중치 최종 병합
merged_model = model.merge_and_unload()

# 병합된 모델 저장
save_directory = "./Qwen-KoAlpaca-Merged"
merged_model.save_pretrained(save_directory)
tokenizer.save_pretrained(save_directory)

print(f"\n[성공] 병합 완료! 좌측 파일 탐색기에서 '{save_directory}' 폴더를 압축(zip)하여 로컬 노트북으로 다운로드하세요.")